<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Attention_Mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
%cd drive/MyDrive

[Errno 2] No such file or directory: 'drive/MyDrive'
/content/drive/MyDrive


In [25]:
import torch
import math
from seq2seq_models import preprocess_text, tokenize_data, pad_or_truncate, Vocab, encode, Embedding
from sequential_models import Linear, Tanh, BasicRNN, LSTMCell, GRUCell

Text preprocessing

In [26]:
dataset_path = "fra.txt"
src, tgt = [], []

with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]

src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (target_label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)

In [ ]:
class RNNEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)

    self.rnn1 = BasicRNN(feature_size, n_neurons)


    #look at hidden state initialization
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    h1 = torch.zeros((x.shape[0], self.neurons))

    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.rnn1(dense_input, h1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    h2 = torch.zeros((output1.shape[0], self.neurons))
    # [batch_size, seq_len, n_hidden], [batch_size, n_hidden]
    output2, h2 = self.rnn2(output1, h2)
    #[num_of_layers, batch_size, n_hidden], [seq_len, batch_size, n_hidden]
    return (h1, h2), output2

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()
    return params

In [27]:
class AdditiveAttention():
  def __init__(self, key_hidden_state, query_hidden_state, new_hidden_state):
    #shape of keys and values = [seq_len, batch_size, n_hidden]
    #hidden state of decoder(query) = [batch_size, n_hidden]
    self.Wk = Linear(key_hidden_state, new_hidden_state)
    self.Wq = Linear(query_hidden_state, new_hidden_state)
    self.Wv = Linear(new_hidden_state, 1)
    self.tanh = Tanh()

  def __repr__(self):
    return f"AdditiveAttention()"

  def __call__(self, keys, queries, values, valid_lens):
    features = self.Wk(keys) + self.Wq(queries)
    #features = [seq_len, batch_size, new_hidden_state]
    scores = self.Wv(self.tanh(features))
    #scores = [seq_len, batch_size, 1]
    scores = scores.permute(1, 2, 0)
    #scores = [batch_size, 1, seq_len]
    values = values.permute(1, 0, 2)
    #values = [batch_size, seq_len, hidden_state]
    self.weights = masked_softmax(scores.squeeze(1), valid_lens)
    self.weights = self.weights.unsqueeze(1)
    #self.weights = [batch_size, 1, seq_len]

    return torch.bmm(self.weights, values)
    #return [batch_size, 1, new_hidden_state]

  def parameters(self):
    return self.Wk.parameters() + self.Wq.parameters() + self.Wv.parameters()

In [ ]:
class RNNAttentionDecoder():
  def __init__(self, feature_size, vocab_size, n_neurons, new_hidden_state, encoder_neurons):
    self.n_neurons = n_neurons
    self.embeddings = Embedding(vocab_size, feature_size)
    self.feature_size = feature_size
    self.vocab_size = vocab_size
    self.rnn1 = BasicRNN(feature_size + encoder_neurons, n_neurons)
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.attention = None
    self.linear = Linear(n_neurons, vocab_size)
    self.new_hidden = new_hidden_state

  def __call__(self, x, h_x, encoder_ouput, x_valid_lens):
    h1, h2 = h_x
    if self.attention is None:
      self.attention = AdditiveAttention(
          encoder_ouput.shape[-1],
          h2.shape[-1],
          self.new_hidden,
      )
    outputs = []
    seq = x.shape[1] #seq_len
    for t in range(seq):
      context = self.attention(encoder_ouput, h2, encoder_ouput, x_valid_lens)

      dense_input = self.embeddings(x[:, t]).unsqueeze(1)

      input_x = torch.cat((dense_input, context), -1)


      output1, h1 = self.rnn1(input_x, h1)
      output1 = output1.permute(1, 0, 2)
      output2, h2 = self.rnn2(output1, h2)

      output2 = output2.squeeze(0)
      outputs.append(output2)

    outputs = torch.stack(outputs)
    outputs = outputs.permute(1, 0, 2)
    logits = self.linear(outputs)
    return logits



  def __repr__(self):
    rep = f"Bahdanau Decoder(\nEmbedding={self.embeddings.embedding.shape},\nBahdanauAttention(), \nRNN=({self.feature_size, self.n_neurons}), \nRNN=({self.n_neurons, self.n_neurons}), \nLinear({self.n_neurons, self.vocab_size}) \n)"
    return rep

  def parameters(self):
    params = self.embeddings.parameters() + self.rnn1.parameters() + self.rnn2.parameters() + self.attention.parameters() + self.linear.parameters()
    return params


In [ ]:
batch_size = 32

#encoder parameters
encoder_vocab_size = len(eng_vocab)
encoder_feature_size = 20
encoder_n_neurons = 10

#decoder parameters
decoder_vocab_size = len(fr_vocab)
decoder_feature_size = 20
decoder_n_neurons = 10
decoder_new_hidden = 20

encoder = RNNEncoder(
    encoder_vocab_size,
    encoder_feature_size,
    encoder_n_neurons
)

decoder = RNNAttentionDecoder(
    decoder_feature_size,
    decoder_vocab_size,
    decoder_n_neurons,
    decoder_new_hidden,
    encoder_n_neurons,
)

decoder.attention = AdditiveAttention(
         encoder_n_neurons,
          decoder_n_neurons,
          decoder_new_hidden,
      )


In [ ]:
import torch.nn.functional as F
epochs = 100
params = encoder.parameters() + decoder.parameters()

for p in params:
  p.requires_grad = True

for epoch in range(epochs):
  idxs = torch.randint(0, batch_size + 1, (batch_size, ))
  #training data
  src_data = src_lookup[idxs]
  #input to decoder
  d_input = decoder_input[idxs]
  #target labels
  labels = target_label[idxs]

  #valid_lengths
  src_lens = src_valid_len[idxs]

  (h1, h2), encoder_output = encoder(src_data)
  #print(logits)
  logits = decoder(d_input, (h1, h2), encoder_output, src_lens)
  b, s, v = logits.shape
  logits = logits.reshape(b*s, -1)
  #use .long() on target values else cross_entropy will not work
  outs = labels.reshape(b*s).long()
  loss = F.cross_entropy(logits, outs)

  for p in params:
    p.grad = None

  loss.backward()

  lr = 0.101

  for p in params:
    p.data += -lr*p.grad

  print(loss.item())






In [77]:
def masked_softmax(X, valid_len):
  """
  Mask <pad> tokens
  """
  def _mask(X, valid_len, value=0):
    #X is 3D, valid is 2D or 1D
    #print(X.shape)
    #print(valid_len.shape)
    maxlen = X.shape[1]
    mask = torch.arange(maxlen) < valid_len.unsqueeze(-1)
    #print(mask.shape)
    X[~mask] = value
    return X

  if valid_len is None:
    return torch.nn.functional.softmax(X, dim=-1)
  else:
    shape = X.shape
    if valid_len.dim() == 1:
      valid_len = valid_len.repeat(shape[1])
    else:
      #flatten
      valid_len = valid_len.reshape(-1)

    X = _mask(X.reshape(-1, X.shape[-1]), valid_len, value=-1e6)
    return torch.nn.functional.softmax(X.reshape(shape), dim=-1)


In [29]:
class DotProductAttention():
  def __init__(self):
    pass

  def __repr__(self):
    return f"DotProductAttention()"

  def __call__(self, queries, keys, values, valid_lens=None):
    d = queries.shape[-1]
    scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
    self.weights = masked_softmax(scores, valid_lens)
    return torch.bmm(self.weights, values)

In [9]:
x = torch.randn((32, 10, 15))
lens = torch.randint(0, 11, (32, ))
z = x.shape[-1]
multi_attention = MultiHeadAttention(z, z, z, 30, 5)
outs = multi_attention(x, x, x, lens)
print(outs.shape)

torch.Size([32, 10, 15])


In [93]:
class PositionalEncoding():
  def __init__(self, num_hiddens, maxlen=1000):
    self.P = torch.zeros((1, maxlen, num_hiddens))
    X = torch.arange(maxlen, dtype=torch.float32).reshape(-1, 1)/torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32)/num_hiddens)
    self.P[:,:, 0::2] = torch.sin(X)
    self.P[:, :, 1::2] = torch.cos(X[:, :self.P[:, :, 1::2].shape[-1]])


  def __call__(self, X):
    return X + self.P[:, :X.shape[1], :]
    pass

  def __repr__(self):
    return "PositionalEncoding()"

  def parameters(self):
    return []


Attention is all you need!

In [31]:
class PositionWiseFFN():
  def __init__(self, input_hidden, ffn_hidden):
    self.l1 = Linear(input_hidden, ffn_hidden)
    self.relu = torch.nn.ReLU()
    self.l2 = Linear(ffn_hidden, input_hidden)


  def __call__(self, X):
    return self.l2(self.relu(self.l1(X)))

  def __repr__(self):
    return "PositionWiseFFN()"

  def parameters(self):
    return self.l1.parameters() + self.l2.parameters()

In [12]:
ffn = PositionWiseFFN(4, 8)
ffn(torch.ones((2, 3, 4)))[0]

tensor([[ 0.3747, -0.1152,  1.3545, -2.0455],
        [ 0.3747, -0.1152,  1.3545, -2.0455],
        [ 0.3747, -0.1152,  1.3545, -2.0455]])

In [32]:
def batch_norm(X,gamma, beta, moving_mean, moving_variance, eps, momentum, train):
  if not train:
    X_hat = (X-moving_mean)/torch.sqrt(moving_variance + eps)
  else:
    mean = X.mean(dim=0)
    var = ((X-mean)**2).mean(dim=0)
    X_hat = (X-mean)/torch.sqrt(var + eps)
    moving_mean = (1-momentum) * moving_mean + momentum * mean
    moving_var = (1-momentum) * moving_mean + momentum * mean
    Y = gamma * X_hat + beta
  return Y, moving_mean, moving_variance

In [33]:
class BatchNorm():
  def __init__(self, num_features):
    shape = (1, num_features)
    self.gamma = torch.ones(shape)
    self.beta = torch.zeros(shape)
    self.moving_mean = torch.zeros(shape)
    self.moving_var = torch.ones(shape)
    self.training = True

  def train(self, mode=True):
    self.training = mode
    return self

  def eval(self):
    return self.train(False)

  def __call__(self, X):
    Y, self.moving_mean, self.moving_var = batch_norm(X, self.gamma, self.beta, self.moving_mean, self.moving_var, 1e-5, 0.1)
    return Y

  def __repr__(self):
    return "BatchNorm()"

  def parameters(self):
    return [self.gamma] + [self.beta]

In [34]:
class Dropout():
  def __init__(self, dropout):
    self.dropout = dropout
    self.training = True

  def train(self, mode=True):
    self.training = mode
    return self

  def eval(self):
    return self.train(False)

  def __call__(self, X):
    if not self.training:
      self.dropout = 0
    if self.dropout == 1:
      return torch.zeros_like(X)
    mask = (torch.rand(X.shape) > self.dropout).float()
    return (mask * X)/(1-self.dropout)

  def __repr__(self):
    return "Dropout()"

  def parameters(self):
    return []


In [35]:
class LayerNorm():
  def __init__(self, num_features):
    shape = (1, num_features)
    self.gamma = torch.ones(shape)
    self.beta = torch.zeros(shape)
    self.eps = 1e-5

  def __call__(self, X):
    mean = torch.mean(X, -1, True)
    var = torch.var(X, -1, keepdim=True)
    X_hat = (X-mean)/torch.sqrt(var+self.eps)
    return self.gamma * X_hat + self.beta


  def __repr__(self):
    return "LayerNorm()"

  def parameters(self):
    return [self.gamma] + [self.beta]

Add + Norm Layer

In [36]:
class AddNorm():
  def __init__(self, num_features, dropout):
    self.l_norm = LayerNorm(num_features)
    self.dropout = Dropout(dropout)

  def __call__(self, X, Y):
    return self.l_norm(self.dropout(Y) + X)

  def train(self, mode=True):
    return self.dropout.train()

  def eval(self):
    return self.dropout.eval()

  def __repr__(self):
    return "AddNorm()"

  def parameters(self):
    return self.l_norm.parameters()

In [18]:
add_norm = AddNorm(4, 0.5)
shape = (2, 3, 4)
add_norm(torch.ones(shape), torch.ones(shape))

tensor([[[-1.5000,  0.5000,  0.5000,  0.5000],
         [ 0.5000,  0.5000, -1.5000,  0.5000],
         [ 1.5000, -0.5000, -0.5000, -0.5000]],

        [[-0.5000, -0.5000,  1.5000, -0.5000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.5000,  0.5000,  0.5000, -1.5000]]])

In [78]:
class MultiHeadAttention():
  def __init__(self, query_size, key_size, value_size, new_hidden, num_heads):
    self.Wq = Linear(query_size, new_hidden)
    self.Wk = Linear(key_size, new_hidden)
    self.Wv = Linear(value_size, new_hidden)
    self.Wo = Linear(new_hidden, value_size)
    self.heads = num_heads
    self.attention = DotProductAttention()

  def transpose_qkv(self, X):
    batch_size, seq_len, hidden_size = X.shape
    #[batch_size, seq_len, num_heads, new_hidden/num_heads]
    X = X.reshape(batch_size, seq_len, self.heads, -1)
    #[batch_size, num_heads, seq_len, new_hidden/num_heads]
    X = X.permute(0, 2, 1, 3)
    #[batch_size*num_heads, seq_len, new_hidden/num_heads]
    X = X.reshape(-1, seq_len, X.shape[3])
    return X

  def transpose_output(self, X):
    #[batch_size*num_heads, seq_len, new_hidden/num_heads]
    #[batch_size, num_head, seq_len, new_hidden/num_heads]
    X = X.reshape(-1, self.heads, X.shape[1], X.shape[2])
    #[batch_size, seq_len, num_heads, new_hidden/num_heads]
    X = X.permute(0, 2, 1, 3)
    #[batch_size,seq_len, new_hidden]
    X = X.reshape(X.shape[0], X.shape[1], -1)
    return X


  def __call__(self, keys, values, queries, valid_lens):
    queries = self.transpose_qkv(self.Wq(queries))
    keys = self.transpose_qkv(self.Wk(keys))
    values = self.transpose_qkv(self.Wv(values))
    if valid_lens is not None:
      if valid_lens.dim() == 1:
        valid_lens = torch.repeat_interleave(valid_lens, self.heads)
      else:
        valid_lens = torch.repeat_interleave(valid_lens, self.heads, dim=0)

    outputs = self.attention(queries, keys, values, valid_lens)
    outputs = self.transpose_output(outputs)
    return self.Wo(outputs)

  def __repr__(self):
    return "MultiHeadAttention()"

  def parameters(self):
    params = self.Wq.parameters() + self.Wk.parameters() + self.Wv.parameters() + self.Wo.parameters()
    return params


In [38]:
x = torch.randn((32, 10, 15))
lens = torch.randint(0, 11, (32, ))
z = x.shape[-1]
encoder = TransformerEncoderBlock(z, z, z, 30, 5, 0.7, z, 40)
encoder(x, lens).shape

torch.Size([32, 10, 15])

In [54]:
class TransformerEncoderBlock():
  def __init__(self, query_size, key_size, value_size, new_hidden, num_heads, dropout, num_features, ffn_hidden):
    self.attention = MultiHeadAttention(query_size, key_size, value_size, new_hidden, num_heads)
    self.addnorm1 = AddNorm(num_features, dropout)
    self.ffn = PositionWiseFFN(num_features, ffn_hidden)
    self.addnorm2 = AddNorm(num_features, dropout)

  def __call__(self, X, valid_lens):
    Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
    return self.addnorm2(Y, self.ffn(Y))

  def __repr__(self):
    x = [f"{value.__repr__()}" for name, value in vars(self).items()]
    return f"TransformerEncoderBlock({x})"

  def train(self):
    x, y = self.addnorm1.train(), self.addnorm2.train()
    return x, y

  def eval(self):
    x, y = self.addnorm1.eval(), self.addnorm2.eval()
    return x, y

  def parameters(self):
    return [value.parameters() for name, value in vars(self).items()]

In [ ]:
#params for transformer encoder
#number of unique tokens in the encoder dataset
vocab_size = len(eng_vocab)

#dimension of vector to represent each token
feature_size = 20

#hidden dimension of the FFN
ffn_hidden = 40

#dimension for projecting the Q-K-Vs into in the attention head
att_hidden = 20

#umber of attention heads
heads = 5

#dropout
dropout = 0.5

#num_blks
blks = 2

#batch_size
batch_size = 32

tr_encoder = TransformerEncoder(vocab_size, feature_size, ffn_hidden, att_hidden, heads, dropout, blks)
x = torch.randn((32, 10, 15))
lens = torch.randint(0, 11, (32, ))
idxs = torch.randint(0, batch_size + 1, (batch_size, ))
#training data
src_data = src_lookup[idxs]
#input to decoder
d_input = decoder_input[idxs]
#target labels
labels = target_label[idxs]
out = tr_encoder(src_data, lens)
tr_encoder

In [59]:
class TransformerEncoder():
  def __init__(self, vocab_size, feature_size, ffn_hidden, att_hidden, num_heads, dropout, num_blks):
    self.training = True
    self.num_hiddens = feature_size
    self.embeddings = Embedding(vocab_size, feature_size)
    self.pos_encoding = PositionalEncoding(feature_size)
    self.blks = [
        TransformerEncoderBlock(feature_size, feature_size, feature_size, att_hidden, num_heads, dropout, feature_size, ffn_hidden ) for _ in range(num_blks)
    ]

  def __call__(self, X, valid_lens):
    X = self.embeddings(X) * math.sqrt(self.num_hiddens)
    self.attention_weights = [None] * len(self.blks)
    for i, blk in enumerate(self.blks):
      X = blk(X, valid_lens)
      self.attention_weights[i] = blk.attention.attention.weights
    return X

  def __repr__(self):
    x = [f"{value.__repr__()}" for name, value in vars(self).items()]
    return f"TransformerEncoder({x[2:]})"

  @property
  def parameters(self):
    return self.embeddings.parameters() + self.pos_encoding.parameters() + [blk.parameters() for blk in self.blks]

  def train(self, mode=True):
    self.training = mode
    return self

  def eval(self):
    return self.train(False)


In [79]:
class DecoderBlock():
  def __init__(self, query_size, key_size, value_size, new_hidden, num_heads, num_features, dropout, ffn_hidden, i):
    self.i = i
    self.attention1 = MultiHeadAttention(query_size, key_size, value_size, new_hidden, num_heads)
    self.addnorm1 = AddNorm(num_features, dropout)
    self.attention2 = MultiHeadAttention(query_size, key_size, value_size, new_hidden, num_heads)
    self.addnorm2 = AddNorm(num_features, dropout)
    self.ffn = PositionWiseFFN(num_features, ffn_hidden)
    self.addnorm3 = AddNorm(num_features, dropout)
    self.training = True

  def __call__(self, X, state):
    enc_outputs, enc_valid_lens = state[0], state[1]
    if state[2][self.i] is None:
      key_values = X
    else:
      key_values = torch.cat((state[2][self.i], X), dim=1)
    state[2][self.i] = key_values
    if self.training:
      batch_size, num_steps, _ = X.shape
      dec_valid_lens = torch.arange(1, num_steps + 1).repeat(batch_size, 1)
    else:
      dec_valid_lens = None

    X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
    Y = self.addnorm1(X, X2)
    Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
    Z = self.addnorm2(Y, Y2)
    return self.addnorm3(Z, self.ffn(Z)), state

  def __repr__(self):
    return f"DecoderBlock{self.i}()"

  def parameters(self):
    models = [self.attention1, self.addorm1, self.attention2, self.addnorm2, self.ffn, self.attention3, self.addnorm3]
    params = [model.parameters() for model in models]
    return params

  def train(self, mode=True):
    self.training = mode
    return self

  def eval(self):
    return self.train(False)

In [84]:
#params for transformer decoder
#number of unique tokens in the decoder dataset
vocab_size = len(eng_vocab)

#dimension of vector to represent each token
feature_size = 20

#hidden dimension of the FFN
ffn_hidden = 40

#dimension for projecting the Q-K-Vs into in the attention head
att_hidden = 20

#umber of attention heads
heads = 5

#dropout
dropout = 0.5

#num_blks
blks = 2

#batch_size
batch_size = 32
x = torch.randn((32, 10, 15))
state = [x, lens, [None]]
print(lens.shape)
print(state[0].shape)
lens = torch.randint(0, 11, (32, ))
z = 15

tr_decoder = DecoderBlock(z, z, z, 30, 5, z, 0.7, 40, 0)
out1, out2 = tr_decoder(x, state)
out1.shape

torch.Size([32])
torch.Size([32, 10, 15])


torch.Size([32, 10, 15])

In [102]:
#params for transformer decoder
#number of unique tokens in the decoder dataset
vocab_size = len(fr_vocab)

#dimension of vector to represent each token
feature_size = 15

#hidden dimension of the FFN
ffn_hidden = 40

#dimension for projecting the Q-K-Vs into in the attention head
att_hidden = 20

#umber of attention heads
heads = 5

#dropout
dropout = 0.5

#num_blks
blks = 2
batch_size = 32
lens = torch.randint(0, 11, (32, ))
idxs = torch.randint(0, batch_size + 1, (batch_size, ))
#training data
src_data = src_lookup[idxs]
#input to decoder
d_input = decoder_input[idxs]

#target labels
labels = target_label[idxs]

# Instantiate the TransformerEncoder for the source data
tr_encoder = TransformerEncoder(
    len(eng_vocab),
    feature_size,
    ffn_hidden,
    att_hidden,
    heads,
    dropout,
    blks
)

# Get valid lengths for the source data
src_lens = src_valid_len[idxs]

# Process src_data through the encoder to get 3D enc_outputs
enc_outputs = tr_encoder(src_data, src_lens)

# Instantiate the TransformerDecoder
tr_decoder = TransformerDecoder(vocab_size, feature_size, blks, feature_size, feature_size, feature_size, feature_size, heads, feature_size, 0.7, ffn_hidden)

# Initialize the decoder state with the encoder outputs
state = tr_decoder.init_state(enc_outputs, src_lens)

out, _ = tr_decoder(d_input, state)
out.shape


torch.Size([32, 15, 21347])

In [94]:
class TransformerDecoder():
  def __init__(self, vocab_size, feature_size, n_blks, query_size, key_size, value_size, new_hidden, num_heads, num_features, dropout, ffn_hidden):
    self.training = True
    self.num_hidden = feature_size
    self.embeddings = Embedding(vocab_size, feature_size)
    self.pos_encoding = PositionalEncoding(feature_size)
    self.n_blks = n_blks
    self.blks = [
        DecoderBlock(query_size, key_size, value_size, new_hidden, num_heads, num_features, dropout, ffn_hidden, i) for i in range(self.n_blks)
    ]
    #find the appropriate dimensions for the linear model
    self.dense = Linear(feature_size, vocab_size)
    self.model = [self.embeddings, self.pos_encoding] + self.blks + [self.dense]

  def init_state(self, enc_outputs, enc_valid_lens):
    return [enc_outputs, enc_valid_lens, [None]*self.n_blks]

  def __call__(self, X, state):
    X = self.pos_encoding(self.embeddings(X) * math.sqrt(self.num_hidden))
    self._attention_weights = [[None] * len(self.blks) for _ in range(2)]
    for i, blk in enumerate(self.blks):
      X, state = blk(X, state)
      #self attention
      self._attention_weights[0][i] = blk.attention1.attention.weights
      #encoder-deocder attention
      self._attention_weights[1][i] = blk.attention2.attention.weights
    return self.dense(X), state

  @property
  def attention_weights(self):
    return self._attention_weights

  def parameters(self):
    return [m.parameters() for m in self.model]

  def __repr__(self):
    return f"AttentionDecoder()"

  def train(self, mode=True):
    self.training = mode
    for blk in self.blks:
      blk.train(mode)
    return self

  def eval(self):
    return self.train(False)